# Publication-safe notebook

This generalized reference implementation expects a live SparkSession named
`spark` and pseudonymous tables matching `docs/data-contract.md`. It contains no
institution-specific ingestion, credentials, internal paths, data, or results.

Parameter defaults are illustrative and intentionally differ from internal
operating settings. Calibrate them only with authorized development data.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import StorageLevel
import numpy as np
import pandas as pd

import os

WRITE_DB = os.environ.get("FRAUD_GRAPH_WRITE_DB", "fraud_graph_demo")
EMP = os.environ.get("FRAUD_GRAPH_PREFIX", "onboarding_fraud")
CUST_COMM  = f"{WRITE_DB}.{EMP}_v2_cust_features_comm"
CUST_CORE  = f"{WRITE_DB}.{EMP}_v2_cust_features"
MOTIF_CUST = f"{WRITE_DB}.{EMP}_v2_motif_cust_features"

CANDIDATES = ["n_shared_mule", "closeness_to_bad", "comm_bad_rate", "n_shared_atm", "ppr",
              "in_deg", "out_deg", "ring_co_feeders", "max_shared_mule_fanin",
              "fraudar_score", "block_density", "in_dense_block", "on_3cycle", "n_3cycle_as_a",
              "passthrough_ratio"]
MOTIF_COLS = ["n_shared_mule_motif", "max_shared_mule_fanin", "ring_co_feeders", "n_shared_atm",
              "on_3cycle", "n_3cycle_as_a", "fraudar_score", "in_dense_block", "block_density",
              "passthrough_ratio"]

K_PCTS = [float(value) for value in os.environ.get("FRAUD_GRAPH_EVALUATION_PCTS", "0.02,0.05").split(",")]

BASELINE_FEATURES = ["n_shared_mule", "closeness_to_bad"]

SELECT_ANCHORS = ["202401", "202402"]
SELECT_KS = [float(value) for value in os.environ.get("FRAUD_GRAPH_SELECTION_PCTS", "0.02,0.05").split(",")]

MIN_GAIN = float(os.environ.get("FRAUD_GRAPH_MIN_GAIN", "0.005"))

RULE_FEATURES = ["in_dense_block", "n_shared_atm", "n_shared_mule"]

OUT_FINAL   = f"{WRITE_DB}.{EMP}_v2_final_features"
OUT_SCORES  = f"{WRITE_DB}.{EMP}_v2_final_scores"
OUT_SELECT  = f"{WRITE_DB}.{EMP}_v2_final_selection"
OUT_VALID   = f"{WRITE_DB}.{EMP}_v2_final_validation"
OUT_SLOPE   = f"{WRITE_DB}.{EMP}_v2_final_sloping"

spark.conf.set("spark.sql.shuffle.partitions", os.environ.get("FRAUD_GRAPH_SHUFFLE_PARTITIONS", "200"))

def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t); df.write.mode("overwrite").saveAsTable(t)

print("final model | select on:", SELECT_ANCHORS, "| select bands:", SELECT_KS, "| K grid:", K_PCTS,
      "| MIN_GAIN:", MIN_GAIN, "| baseline seed:", BASELINE_FEATURES)


In [ ]:
def _table_exists(t):
    try:
        spark.table(t).limit(0).count(); return True
    except Exception:
        return False

BASE_TBL = CUST_COMM if _table_exists(CUST_COMM) else CUST_CORE
print("base feature table:", BASE_TBL, "(comm_bad_rate available)" if BASE_TBL == CUST_COMM else "(FALLBACK — comm_bad_rate=0; run 04b for the community feature)")

base = spark.table(BASE_TBL)
if "comm_bad_rate" not in base.columns:
    base = base.withColumn("comm_bad_rate", F.lit(0.0))

_motif_have = [c for c in MOTIF_COLS if c in spark.table(MOTIF_CUST).columns]
motif = spark.table(MOTIF_CUST).select("cust", "anchor_month", *_motif_have)

full = (base.join(motif, ["cust", "anchor_month"], "left")
        .na.fill({c: 0 for c in MOTIF_COLS}).na.fill({"fraudar_score": 0.0, "block_density": 0.0, "passthrough_ratio": 0.0,
                                                       "comm_bad_rate": 0.0, "ppr": 0.0, "closeness_to_bad": 0.0}))

for c in CANDIDATES:
    if c not in full.columns:
        full = full.withColumn(c, F.lit(0.0))
full = full.na.fill({c: 0 for c in CANDIDATES}).withColumn("target_fraud", F.col("target_fraud").cast("int"))
save_overwrite(full, OUT_FINAL)
print("saved master feature table:", OUT_FINAL, "| rows:", spark.table(OUT_FINAL).count(),
      "| anchors:", sorted([r[0] for r in full.select("anchor_month").distinct().collect()]))


In [ ]:
pdf = (spark.table(OUT_FINAL)
       .select("cust", "anchor_month", "target_fraud", *CANDIDATES).toPandas())
pdf["target_fraud"] = pdf["target_fraud"].fillna(0).astype(int)
for c in CANDIDATES:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(0.0)
ANCHORS = sorted(pdf["anchor_month"].unique().tolist())
print("pandas frame:", pdf.shape, "| anchors:", ANCHORS,
      "| bad by anchor:", {a: int(pdf.loc[pdf.anchor_month == a, "target_fraud"].sum()) for a in ANCHORS})

def capture_at_pd(y, score, ks):

    y = np.asarray(y, dtype=int); score = np.asarray(score, dtype=float)
    order = np.argsort(-score, kind="mergesort")
    ys = y[order]; n = len(y); nbad = int(y.sum()); rows = []
    for p in ks:
        K = max(1, int(n * p)); topbad = int(ys[:K].sum())
        cap = (topbad / nbad) if nbad else 0.0
        lift = ((topbad / K) / (nbad / n)) if nbad else 0.0
        rows.append((p, K, topbad, cap, lift))
    return rows, n, nbad

def _woe_iv(cnt, bad):
    good = cnt - bad; tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6) if tb else bad * 0 + 1e-6
    pg = (good / tg).clip(lower=1e-6) if tg else good * 0 + 1e-6
    return float(((pg - pb) * np.log(pg / pb)).sum())

def _robust_bins(x, max_bins=10):
    s = pd.Series(np.asarray(x, dtype=float)); nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1: return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0); out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"), q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values; return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values


In [ ]:
def _pctrank_group(s):
    n = len(s)
    if n <= 1:
        return pd.Series(np.zeros(n), index=s.index)
    return (s.rank(method="min") - 1.0) / (n - 1.0)

PR = {}
for c in CANDIDATES:
    PR[c] = pdf.groupby("anchor_month")[c].transform(_pctrank_group).fillna(0.0).values
pr_df = pd.DataFrame(PR); pr_df["anchor_month"] = pdf["anchor_month"].values
pr_df["target_fraud"] = pdf["target_fraud"].values

def mean_capture(cols, ks, anchors):

    ks = ks if isinstance(ks, (list, tuple)) else [ks]
    caps = []
    for a in anchors:
        d = pr_df[pr_df.anchor_month == a]
        score = d[list(cols)].sum(axis=1).values if cols else np.zeros(len(d))
        rows, _, _ = capture_at_pd(d["target_fraud"].values, score, ks)
        caps.append(float(np.mean([r[3] for r in rows])))
    return float(np.mean(caps)) if caps else 0.0

SEL_ANCHORS = [a for a in SELECT_ANCHORS if a in ANCHORS] or ANCHORS
print("selection on anchors:", SEL_ANCHORS, "| validation on:", ANCHORS, "| baseline seed:", BASELINE_FEATURES)

solo = sorted([(mean_capture([c], SELECT_KS, SEL_ANCHORS), c) for c in CANDIDATES], reverse=True)
print("=== single-feature mean capture@%s (mature anchors) ===" % ("/".join("%.0f%%" % (k * 100) for k in SELECT_KS)))
for m, c in solo:
    print("   %-22s %.4f%s" % (c, m, "   <- 04-core (seeded)" if c in BASELINE_FEATURES else ""))


In [ ]:
def _loao_capture(cols):
    return float(np.mean([mean_capture(cols, SELECT_KS, [held]) for held in ANCHORS])) if ANCHORS else 0.0

if "ppr" in CANDIDATES and "closeness_to_bad" in CANDIDATES:
    bb_close, bb_ppr = ["n_shared_mule", "closeness_to_bad"], ["n_shared_mule", "ppr"]
    print("=== proximity backbone A/B (mean capture@%s) ===" % "/".join("%.0f%%" % (k * 100) for k in SELECT_KS))
    print("   solo:  closeness_to_bad=%.4f   ppr=%.4f" %
          (mean_capture(["closeness_to_bad"], SELECT_KS, SEL_ANCHORS), mean_capture(["ppr"], SELECT_KS, SEL_ANCHORS)))
    print("   backbone {n_shared_mule + closeness}: mature=%.4f  LOAO=%.4f" %
          (mean_capture(bb_close, SELECT_KS, SEL_ANCHORS), _loao_capture(bb_close)))
    print("   backbone {n_shared_mule + ppr}      : mature=%.4f  LOAO=%.4f" %
          (mean_capture(bb_ppr, SELECT_KS, SEL_ANCHORS), _loao_capture(bb_ppr)))
    print("   READ: if closeness >= ppr (esp. LOAO) -> keeping closeness & dropping ppr is justified on evidence;")
    print("         if ppr > closeness out-of-sample -> switch backbone to ppr (set BASELINE_FEATURES accordingly).")


In [ ]:
included = [c for c in BASELINE_FEATURES if c in CANDIDATES]
remaining = [c for _, c in solo if c not in included]
best_metric = mean_capture(included, SELECT_KS, SEL_ANCHORS)
select_log = [(0, c, "BASELINE", round(best_metric, 4), 0.0) for c in included]
print("=== forward selection OVER 04-core (mature anchors) ===")
print("   baseline %-28s mean_cap=%.4f" % ("+".join(included), best_metric))
step = 0
while remaining:
    scored = sorted([(mean_capture(included + [c], SELECT_KS, SEL_ANCHORS), c) for c in remaining], reverse=True)
    cand_metric, cand = scored[0]; gain = cand_metric - best_metric
    if gain > MIN_GAIN:
        step += 1; included.append(cand); remaining.remove(cand); best_metric = cand_metric
        select_log.append((step, cand, "ADD", round(cand_metric, 4), round(gain, 4)))
        print("   step %d: ADD  %-22s mean_cap=%.4f  (+%.4f)" % (step, cand, cand_metric, gain))
    else:
        select_log.append((step + 1, cand, "STOP", round(cand_metric, 4), round(gain, 4)))
        print("   stop: best remaining add %-18s would give %.4f (gain %.4f <= MIN_GAIN %.3f) -> not added"
              % (cand, cand_metric, gain, MIN_GAIN))
        break

SELECTED = list(included)
print("\nSELECTED feature set (%d):" % len(SELECTED), SELECTED)
print("   = 04-core %s + motifs %s" % (BASELINE_FEATURES, [c for c in SELECTED if c not in BASELINE_FEATURES]))
print("selection mean capture (mature, %s): %.4f" % ("/".join("%.0f%%" % (k * 100) for k in SELECT_KS), best_metric))
sel_sdf = spark.createDataFrame([(int(s), f, a, float(m), float(g)) for (s, f, a, m, g) in select_log],
                                ["step", "feature", "action", "mean_capture_sel", "marginal_gain"])
save_overwrite(sel_sdf, OUT_SELECT)
print("saved selection log:", OUT_SELECT)


In [ ]:
def forward_select(anchors, ks, min_gain):
    inc = [c for c in BASELINE_FEATURES if c in CANDIDATES]
    rem = [c for c in CANDIDATES if c not in inc]; bm = mean_capture(inc, ks, anchors)
    while rem:
        sc = sorted([(mean_capture(inc + [c], ks, anchors), c) for c in rem], reverse=True)
        m, c = sc[0]
        if m - bm > min_gain:
            inc.append(c); rem.remove(c); bm = m
        else:
            break
    return inc

if len(ANCHORS) >= 2:
    for held in ANCHORS:
        train = [a for a in ANCHORS if a != held]
        sel_t = forward_select(train, SELECT_KS, MIN_GAIN)
        oot = mean_capture(sel_t, SELECT_KS, [held])
        insample = mean_capture(sel_t, SELECT_KS, train)
        motifs_t = [c for c in sel_t if c not in BASELINE_FEATURES]
        print("   held-out %s: trained-on-%s -> in-sample=%.4f | HELD-OUT=%.4f | added motifs=%s"
              % (held, "+".join(train), insample, oot, motifs_t))
else:
    print("   (only one anchor present — run with TEST_ONE_ANCHOR=False upstream for the LOAO check)")


In [ ]:
if not SELECTED:
    SELECTED = [solo[0][1]]; print("   [WARN] selection empty — using the single feature with the highest validation metric:", SELECTED)
sel_pr_cols = SELECTED
pdf["final_score"] = pr_df[sel_pr_cols].mean(axis=1).values
pdf["risk_pctl"] = pdf.groupby("anchor_month")["final_score"].rank(method="first", pct=True).values * 100.0

scores_pd = pdf[["cust", "anchor_month", "target_fraud", "final_score", "risk_pctl"] + SELECTED].copy()
scores_pd["target_fraud"] = scores_pd["target_fraud"].astype(int)
scores_pd["final_score"] = scores_pd["final_score"].astype(float)
scores_pd["risk_pctl"] = scores_pd["risk_pctl"].astype(float)
for c in SELECTED:
    scores_pd[c] = scores_pd[c].astype(float)
scores_sdf = spark.createDataFrame(scores_pd)
save_overwrite(scores_sdf, OUT_SCORES)
print("saved final scores:", OUT_SCORES, "| rows:", scores_sdf.count(), "| features in score:", SELECTED)


In [ ]:
BASELINE = [c for c in ["closeness_to_bad", "n_shared_mule"] if c in CANDIDATES]
def score_of(cols, a):
    d = pr_df[pr_df.anchor_month == a]
    return d[list(cols)].sum(axis=1).values, d["target_fraud"].values

valid_rows = []
models = [("04_core_baseline", BASELINE), ("FINAL_selected", SELECTED)]
for mname, cols in models:
    agg = {p: [] for p in K_PCTS}; aggl = {p: [] for p in K_PCTS}
    for a in ANCHORS:
        sc, y = score_of(cols, a)
        rows, n, nbad = capture_at_pd(y, sc, K_PCTS)
        for (p, K, topbad, cap, lift) in rows:
            valid_rows.append((mname, a, p, K, n, nbad, topbad, round(cap, 4), round(lift, 3)))
            agg[p].append(cap); aggl[p].append(lift)
    for p in K_PCTS:
        valid_rows.append((mname, "MEAN", p, -1, -1, -1, -1,
                           round(float(np.mean(agg[p])), 4), round(float(np.mean(aggl[p])), 3)))

print("=== capture@K : 04_core baseline vs FINAL selected ===")
for (mname, scope, p, K, n, nbad, topbad, cap, lift) in valid_rows:
    if scope == "MEAN":
        print("   %-16s  MEAN     top%4.0f%%  capture=%.4f  lift=%.2f" % (mname, p * 100, cap, lift))
valid_sdf = spark.createDataFrame(valid_rows, ["model", "scope", "top_pct", "K", "n_cust", "n_bad",
                                               "bad_in_topK", "capture", "lift"])
save_overwrite(valid_sdf, OUT_VALID)
print("saved validation table:", OUT_VALID)


In [ ]:
slope_rows = []
for a in ANCHORS:
    d = pdf[pdf.anchor_month == a]; y = d["target_fraud"].astype(int).values; tot_bad = int(y.sum())
    b = _robust_bins(d["final_score"].astype(float).values, max_bins=10)
    g = pd.DataFrame({"b": b, "y": y}).groupby("b").agg(cnt=("y", "size"), bad=("y", "sum")).sort_index()
    g["event_rate_pct"] = 100.0 * g["bad"] / g["cnt"]
    g["pct_fraud_captured"] = (100.0 * g["bad"] / tot_bad) if tot_bad else 0.0
    iv = _woe_iv(g["cnt"], g["bad"]); er = g["event_rate_pct"].values
    mono = bool(np.all(np.diff(er) >= -1e-9) or np.all(np.diff(er) <= 1e-9))
    print("=== final_score sloping @ %s (bad=%d / n=%d)  IV=%.3f  bins=%d  monotonic=%s  top-bin event=%.2f%% ==="
          % (a, tot_bad, len(d), iv, len(g), mono, er[-1] if len(er) else 0))
    for bi, r in g.reset_index().iterrows():
        slope_rows.append((a, "final_score", int(r["b"]), int(r["cnt"]), int(r["bad"]),
                           round(float(r["event_rate_pct"]), 4), round(float(r["pct_fraud_captured"]), 4),
                           round(iv, 4), mono))
sdf = spark.createDataFrame(slope_rows, ["anchor_month", "feature", "bin", "cnt", "bad",
                                         "event_rate_pct", "pct_fraud_captured", "iv", "monotonic"])
save_overwrite(sdf, OUT_SLOPE)
print("saved final sloping:", OUT_SLOPE)


In [ ]:
print("=== rule overlays (pooled across anchors) ===")
tot_bad_all = int(pdf["target_fraud"].sum()); n_all = len(pdf)
print("   base rate: %d bad / %d customers = %.3f%%" % (tot_bad_all, n_all, 100.0 * tot_bad_all / max(n_all, 1)))
for rf in RULE_FEATURES:
    if rf not in pdf.columns: continue
    flag = pdf[rf].astype(float) > 0
    nf = int(flag.sum()); bf = int(pdf.loc[flag, "target_fraud"].sum())
    prec = (bf / nf) if nf else 0.0; rec = (bf / tot_bad_all) if tot_bad_all else 0.0
    print("   %-16s > 0 : flags %6d cust | %4d fraud | precision=%.2f%% | recall=%.2f%% | lift=%.1fx"
          % (rf, nf, bf, 100.0 * prec, 100.0 * rec, (prec / (tot_bad_all / max(n_all, 1))) if tot_bad_all else 0.0))


In [ ]:
print("=== highest-risk customers preview (top by risk_pctl, per anchor) ===")
(spark.table(OUT_SCORES).withColumn("rk", F.row_number().over(
        Window.partitionBy("anchor_month").orderBy(F.col("risk_pctl").desc())))
    .where(F.col("rk") <= 5).drop("rk")
    .select("anchor_month", "cust", F.round("risk_pctl", 2).alias("risk_pctl"),
            F.round("final_score", 4).alias("final_score"), "target_fraud")
    .orderBy("anchor_month", F.col("risk_pctl").desc()).show(30, False))

print("=== Notebook 06 (v2) summary ===")
print("master features :", OUT_FINAL,  "| rows:", spark.table(OUT_FINAL).count())
print("final scores    :", OUT_SCORES, "| rows:", spark.table(OUT_SCORES).count())
print("selected features:", SELECTED)
print("selection log   :", OUT_SELECT, "| validation:", OUT_VALID, "| sloping:", OUT_SLOPE)
print("Example triage = rank by risk_pctl and review configured rule flags under human oversight.")
print("NOTEBOOK 06 (v2) COMPLETE — end-to-end example score built by validation-driven forward selection.")


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import IPython.display as _disp
from pyspark.sql import functions as _F

CUST_TBL      = f"{WRITE_DB}.{EMP}_v2_final_scores"
PLOT_FEATURES = ["final_score"]
PLOT_ANCHOR   = "202401"
MIN_BIN_FRAC = float(os.environ.get("FRAUD_GRAPH_MIN_BIN_FRACTION", "0.02"))
MIN_STABLE_CNT = int(os.environ.get("FRAUD_GRAPH_MIN_STABLE_COUNT", "30"))
MIN_STABLE_BAD = int(os.environ.get("FRAUD_GRAPH_MIN_STABLE_POSITIVES", "3"))
RISKY_ON_LEFT = True

def _wilson_lcb(bad, cnt, z=1.96):
    bad = np.asarray(bad, float); cnt = np.asarray(cnt, float)
    p = np.where(cnt > 0, bad / cnt, 0.0); denom = 1 + z*z/np.maximum(cnt, 1)
    centre = p + z*z/(2*np.maximum(cnt, 1)); margin = z*np.sqrt((p*(1-p) + z*z/(4*np.maximum(cnt,1)))/np.maximum(cnt,1))
    return np.where(cnt > 0, np.clip((centre - margin)/denom, 0, 1), 0.0)

def _robust_bins(x, max_bins=10):
    s = pd.Series(np.asarray(x, dtype=float)); nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1: return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0); out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"), q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values; return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values

def _club_sparse(d, n, min_frac):
    d = d.sort_index().copy()
    while len(d) > 2:
        share = d["cnt"] / n
        if share.min() >= min_frac: break
        j = int(share.values.argmin()); k = j + 1 if j == 0 else j - 1
        lo, hi = min(j, k), max(j, k)
        d.loc[d.index[lo], ["cnt", "bad"]] += d.loc[d.index[hi], ["cnt", "bad"]].values
        d.loc[d.index[lo], "hi"] = max(d.loc[d.index[lo], "hi"], d.loc[d.index[hi], "hi"])
        d.loc[d.index[lo], "lo"] = min(d.loc[d.index[lo], "lo"], d.loc[d.index[hi], "lo"])
        d = d.drop(d.index[hi]).reset_index(drop=True)
    return d

def _woe_iv(cnt, bad):
    good = cnt - bad; tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6); pg = (good / tg).clip(lower=1e-6)
    return float(((pg - pb) * np.log(pg / pb)).sum())

_pdf = spark.table(CUST_TBL).where(_F.col("anchor_month") == PLOT_ANCHOR).select("target_fraud", *PLOT_FEATURES).toPandas()
_y = _pdf["target_fraud"].astype(int).values; _tot_bad = int(_y.sum()); _tot_good = len(_y) - _tot_bad
print("plotting anchor %s | customers=%d | frauds=%d" % (PLOT_ANCHOR, len(_pdf), _tot_bad))

def plot_sloping(feature):
    x = _pdf[feature].astype(float).values
    d = (pd.DataFrame({"b": _robust_bins(x), "x": x, "y": _y}).groupby("b")
         .agg(cnt=("y", "size"), bad=("y", "sum"), lo=("x", "min"), hi=("x", "max")).sort_index())
    d = _club_sparse(d, len(_pdf), MIN_BIN_FRAC)
    d = (d.iloc[::-1] if RISKY_ON_LEFT else d).reset_index(drop=True)
    d["event_rate_pct"] = 100.0 * d["bad"] / d["cnt"]
    cum_bad = d["bad"].cumsum(); cum_good = (d["cnt"] - d["bad"]).cumsum()
    d["pct_captured"] = 100.0 * cum_bad / max(_tot_bad, 1)
    ks = float((100.0 * cum_bad / max(_tot_bad, 1) - 100.0 * cum_good / max(_tot_good, 1)).abs().max())
    iv = _woe_iv(d["cnt"], d["bad"])
    d["er_lcb_pct"] = 100.0 * _wilson_lcb(d["bad"].values, d["cnt"].values)
    d["stable"] = (d["cnt"] >= MIN_STABLE_CNT) & (d["bad"] >= MIN_STABLE_BAD)
    labels = [("%.3g" % lo) if lo == hi else ("[%.3g, %.3g]" % (lo, hi)) for lo, hi in zip(d["lo"], d["hi"])]
    xs = range(len(d))
    fig, ax1 = plt.subplots(figsize=(11, 5))
    ax1.bar(xs, d["event_rate_pct"], color="#c97b84", alpha=0.85, label="Event Rate (%)")
    for i, v in enumerate(d["event_rate_pct"]): ax1.text(i, v, "%.2f" % v, ha="center", va="bottom", fontsize=8)
    ax1.set_ylabel("Event Rate (%)"); ax1.set_xticks(list(xs)); ax1.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    for i, st in enumerate(d["stable"].values):
        if not st: ax1.patches[i].set_hatch("///"); ax1.patches[i].set_edgecolor("#7a2a34")
    ax2 = ax1.twinx()
    ax2.plot(xs, d["pct_captured"], color="#a23b4e", marker="o", label="Cumulative % Fraud Captured")
    ax2.set_ylabel("Cumulative % Fraud Captured"); ax2.set_ylim(0, 105)
    ax1.set_title("%s @ %s   (IV=%.3f, KS=%.1f)   hatched = UNSTABLE (n<%d)\nEvent Rate (bars) & Cumulative %% Fraud Captured (line)"
                  % (feature, PLOT_ANCHOR, iv, ks, MIN_STABLE_CNT), fontsize=12)
    ax1.legend(loc="upper left"); ax2.legend(loc="lower right"); fig.tight_layout(); plt.show()
    _disp.display(pd.DataFrame({"bin": labels, "# frauds": d["bad"].astype(int).values,
                                "# customers": d["cnt"].astype(int).values,
                                "event_rate_%": d["event_rate_pct"].round(3).values,
                                "event_rate_LCB_%": d["er_lcb_pct"].round(2).values,
                                "cum_%_captured": d["pct_captured"].round(2).values,
                                "stable": d["stable"].values}))

for _f in PLOT_FEATURES:
    try: plot_sloping(_f)
    except Exception as _e: print("  %s: skipped (%s)" % (_f, _e))


In [ ]:
FEAT_TBL     = f"{WRITE_DB}.{EMP}_v2_final_features"
AUDIT_ANCHOR = "202401"
HI_CORR = float(os.environ.get("FRAUD_GRAPH_HIGH_CORRELATION", "0.75"))
_IDC         = {"id", "anchor_month", "target_fraud", "is_candidate", "is_known_bad", "cust"}

_pa = spark.table(FEAT_TBL).where(_F.col("anchor_month") == AUDIT_ANCHOR).toPandas()
_fc = [c for c in _pa.columns if c not in _IDC and pd.api.types.is_numeric_dtype(_pa[c])]
_fc = [c for c in _fc if _pa[c].nunique(dropna=True) > 1]
_yv = _pa["target_fraud"].astype(int).values if "target_fraud" in _pa else None
_XA = _pa[_fc].astype(float).fillna(0.0)
print("audit anchor %s | customers=%d | features=%d | frauds=%d" %
      (AUDIT_ANCHOR, len(_pa), len(_fc), int(_yv.sum()) if _yv is not None else -1))

_spm = _XA.corr(method="spearman")
print("\n--- Spearman correlation ---\n", _spm.round(2).to_string())

def _jacc(a, b):
    a = a.values > 0; b = b.values > 0; u = (a | b).sum()
    return float((a & b).sum() / u) if u else 0.0

print("\n--- REDUNDANT pairs (|Spearman| > %.2f): keep only ONE ---" % HI_CORR)
_found = False
for _i in range(len(_fc)):
    for _j in range(_i + 1, len(_fc)):
        _r = _spm.iloc[_i, _j]
        if pd.notna(_r) and abs(_r) > HI_CORR:
            _found = True
            print("  %-22s ~ %-22s  Spearman=%+.2f  nonzero-overlap=%.2f"
                  % (_fc[_i], _fc[_j], _r, _jacc(_XA[_fc[_i]], _XA[_fc[_j]])))
if not _found: print("  none — features largely independent.")

def _iv10(x, yv):
    s = pd.Series(x)
    if s.nunique() <= 1: return 0.0
    try: b = pd.qcut(s.rank(method="first"), 10, labels=False, duplicates="drop")
    except Exception: b = pd.Series(np.zeros(len(s)))
    g = pd.DataFrame({"b": b, "y": yv}).groupby("b").agg(cnt=("y", "size"), bad=("y", "sum"))
    good = g["cnt"] - g["bad"]
    pb = (g["bad"] / max(g["bad"].sum(), 1)).clip(1e-6); pg = (good / max(good.sum(), 1)).clip(1e-6)
    return float(((pg - pb) * np.log(pg / pb)).sum())

_rows = []
for _c in _fc:
    _x = _XA[_c].values
    _sc = float(pd.Series(_x).corr(pd.Series(_yv), method="spearman")) if _yv is not None else np.nan
    _rows.append((_c, round(100.0*(_x==0).mean(),1), round(_sc,3) if _sc==_sc else None, round(_iv10(_x,_yv),3)))
_aud = pd.DataFrame(_rows, columns=["feature","zeros_%","spearman_vs_target","IV"]).sort_values("IV", ascending=False)
print("\n--- FAIR per-feature strength (sorted by IV) ---\n", _aud.to_string(index=False))
